In [1]:
import pandas as pd
from pathlib import Path

X_train = pd.read_csv("../../data/processed/X_train.csv")
X_test = pd.read_csv("../../data/processed/X_test.csv")

y_train = pd.read_csv("../../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../../data/processed/y_test.csv").squeeze()

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (24000, 34)
X_test : (6000, 34)
y_train: (24000,)
y_test : (6000,)


In [2]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

In [3]:
rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

rf_param_dist = {
    "n_estimators": [100, 150, 200, 250],
    "max_depth": [None, 10, 15, 20, 25, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

rf_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=rf_param_dist,
    n_iter=15,
    scoring="f1",
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

rf_search.fit(X_train, y_train)

Fitting 3 folds for each of 15 candidates, totalling 45 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'max_depth': [None, 10, ...], 'max_features': ['sqrt', 'log2'], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",15
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichret

In [4]:
print("Best Random Forest Parameters:")
print(rf_search.best_params_)

print("\nBest Cross-Validation F1 Score:")
print(round(rf_search.best_score_, 4))

Best Random Forest Parameters:
{'n_estimators': 250, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None}

Best Cross-Validation F1 Score:
0.9327


In [5]:
best_rf = rf_search.best_estimator_

y_pred_rf_tuned = best_rf.predict(X_test)

rf_tuned_accuracy = accuracy_score(y_test, y_pred_rf_tuned)
rf_tuned_precision = precision_score(y_test, y_pred_rf_tuned)
rf_tuned_recall = recall_score(y_test, y_pred_rf_tuned)
rf_tuned_f1 = f1_score(y_test, y_pred_rf_tuned)

print("Tuned Random Forest")
print("-------------------")
print("Accuracy :", round(rf_tuned_accuracy, 4))
print("Precision:", round(rf_tuned_precision, 4))
print("Recall   :", round(rf_tuned_recall, 4))
print("F1 Score :", round(rf_tuned_f1, 4))

Tuned Random Forest
-------------------
Accuracy : 0.9338
Precision: 0.9569
Recall   : 0.9161
F1 Score : 0.9361


In [6]:
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf_tuned))


Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.95      0.93      2828
           1       0.96      0.92      0.94      3172

    accuracy                           0.93      6000
   macro avg       0.93      0.93      0.93      6000
weighted avg       0.93      0.93      0.93      6000



In [7]:
et = ExtraTreesClassifier(
    random_state=42,
    n_jobs=-1
)

et_param_dist = {
    "n_estimators": [100, 150, 200, 250],
    "max_depth": [None, 10, 15, 20, 25, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

et_search = RandomizedSearchCV(
    estimator=et,
    param_distributions=et_param_dist,
    n_iter=15,
    scoring="f1",
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

et_search.fit(X_train, y_train)

Fitting 3 folds for each of 15 candidates, totalling 45 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",ExtraTreesCla...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'max_depth': [None, 10, ...], 'max_features': ['sqrt', 'log2'], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",15
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichret

In [8]:
print("Best Extra Trees Parameters:")
print(et_search.best_params_)

print("\nBest Cross-Validation F1 Score:")
print(round(et_search.best_score_, 4))

Best Extra Trees Parameters:
{'n_estimators': 250, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None}

Best Cross-Validation F1 Score:
0.9352


In [9]:
best_et = et_search.best_estimator_

y_pred_et_tuned = best_et.predict(X_test)

et_tuned_accuracy = accuracy_score(y_test, y_pred_et_tuned)
et_tuned_precision = precision_score(y_test, y_pred_et_tuned)
et_tuned_recall = recall_score(y_test, y_pred_et_tuned)
et_tuned_f1 = f1_score(y_test, y_pred_et_tuned)

print("Tuned Extra Trees")
print("-----------------")
print("Accuracy :", round(et_tuned_accuracy, 4))
print("Precision:", round(et_tuned_precision, 4))
print("Recall   :", round(et_tuned_recall, 4))
print("F1 Score :", round(et_tuned_f1, 4))

Tuned Extra Trees
-----------------
Accuracy : 0.9318
Precision: 0.951
Recall   : 0.9183
F1 Score : 0.9344


In [10]:
print("\nClassification Report:")
print(classification_report(y_test, y_pred_et_tuned))


Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.95      0.93      2828
           1       0.95      0.92      0.93      3172

    accuracy                           0.93      6000
   macro avg       0.93      0.93      0.93      6000
weighted avg       0.93      0.93      0.93      6000



In [11]:
tuning_results = pd.DataFrame({
    "Model": [
        "Random Forest - Baseline",
        "Random Forest - Tuned",
        "Extra Trees - Baseline",
        "Extra Trees - Tuned"
    ],
    
    "Accuracy": [
        0.9350,
        rf_tuned_accuracy,
        0.9315,
        et_tuned_accuracy
    ],
    
    "Precision": [
        0.9624,
        rf_tuned_precision,
        0.9498,
        et_tuned_precision
    ],
    
    "Recall": [
        0.9127,
        rf_tuned_recall,
        0.9190,
        et_tuned_recall
    ],
    
    "F1 Score": [
        0.9369,
        rf_tuned_f1,
        0.9341,
        et_tuned_f1
    ]
})

tuning_results

,Model,Accuracy,Precision,Recall,F1 Score
0,Random Forest - Baseline,0.935000,0.962400,0.912700,0.936900
1,Random Forest - Tuned,0.933833,0.956865,0.916141,0.936061
2,Extra Trees - Baseline,0.931500,0.949800,0.919000,0.934100
3,Extra Trees - Tuned,0.931833,0.951028,0.918348,0.934403


In [12]:
display_results = tuning_results.copy()

for column in ["Accuracy", "Precision", "Recall", "F1 Score"]:
    display_results[column] = (
        display_results[column] * 100
    ).round(2).astype(str) + "%"

display_results

,Model,Accuracy,Precision,Recall,F1 Score
0,Random Forest - Baseline,93.5%,96.24%,91.27%,93.69%
1,Random Forest - Tuned,93.38%,95.69%,91.61%,93.61%
2,Extra Trees - Baseline,93.15%,94.98%,91.9%,93.41%
3,Extra Trees - Tuned,93.18%,95.1%,91.83%,93.44%


In [13]:
best_row = tuning_results.loc[
    tuning_results["F1 Score"].idxmax()
]

print("Best Model:")
print(best_row["Model"])

print("\nF1 Score:", f"{best_row['F1 Score']:.2%}")
print("Accuracy:", f"{best_row['Accuracy']:.2%}")
print("Precision:", f"{best_row['Precision']:.2%}")
print("Recall:", f"{best_row['Recall']:.2%}")

Best Model:
Random Forest - Baseline

F1 Score: 93.69%
Accuracy: 93.50%
Precision: 96.24%
Recall: 91.27%


In [14]:
from pathlib import Path

reports_dir = Path("../../reports")
reports_dir.mkdir(parents=True, exist_ok=True)

tuning_results.to_csv(
    reports_dir / "hyperparameter_tuning_results.csv",
    index=False
)

print("Tuning results saved successfully.")

Tuning results saved successfully.


In [15]:
import joblib
from pathlib import Path

models_dir = Path("../../models")
models_dir.mkdir(parents=True, exist_ok=True)

if rf_tuned_f1 >= et_tuned_f1:
    final_model = best_rf
    final_model_name = "random_forest_tuned.pkl"
else:
    final_model = best_et
    final_model_name = "extra_trees_tuned.pkl"

joblib.dump(
    final_model,
    models_dir / final_model_name
)

print("Best model saved as:")
print(final_model_name)

Best model saved as:
random_forest_tuned.pkl


In [16]:
best_rf = rf_search.best_estimator_

y_pred_rf_tuned = best_rf.predict(X_test)

rf_tuned_accuracy = accuracy_score(y_test, y_pred_rf_tuned)
rf_tuned_precision = precision_score(y_test, y_pred_rf_tuned)
rf_tuned_recall = recall_score(y_test, y_pred_rf_tuned)
rf_tuned_f1 = f1_score(y_test, y_pred_rf_tuned)

print("Tuned Random Forest")
print("-------------------")
print("Accuracy :", round(rf_tuned_accuracy, 4))
print("Precision:", round(rf_tuned_precision, 4))
print("Recall   :", round(rf_tuned_recall, 4))
print("F1 Score :", round(rf_tuned_f1, 4))

Tuned Random Forest
-------------------
Accuracy : 0.9338
Precision: 0.9569
Recall   : 0.9161
F1 Score : 0.9361


In [17]:
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf_tuned))


Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.95      0.93      2828
           1       0.96      0.92      0.94      3172

    accuracy                           0.93      6000
   macro avg       0.93      0.93      0.93      6000
weighted avg       0.93      0.93      0.93      6000



In [18]:
print("Tuned Random Forest")
print("-------------------")
print("Accuracy :", round(rf_tuned_accuracy, 4))
print("Precision:", round(rf_tuned_precision, 4))
print("Recall   :", round(rf_tuned_recall, 4))
print("F1 Score :", round(rf_tuned_f1, 4))

Tuned Random Forest
-------------------
Accuracy : 0.9338
Precision: 0.9569
Recall   : 0.9161
F1 Score : 0.9361


In [19]:
rf_comparison = pd.DataFrame({
    "Model": [
        "Random Forest - Baseline",
        "Random Forest - Tuned"
    ],
    "Accuracy": [
        0.9350,
        rf_tuned_accuracy
    ],
    "Precision": [
        0.9624,
        rf_tuned_precision
    ],
    "Recall": [
        0.9127,
        rf_tuned_recall
    ],
    "F1 Score": [
        0.9369,
        rf_tuned_f1
    ]
})

rf_comparison

,Model,Accuracy,Precision,Recall,F1 Score
0,Random Forest - Baseline,0.935000,0.962400,0.912700,0.936900
1,Random Forest - Tuned,0.933833,0.956865,0.916141,0.936061


In [20]:
rf_display = rf_comparison.copy()

for col in ["Accuracy", "Precision", "Recall", "F1 Score"]:
    rf_display[col] = (
        rf_display[col] * 100
    ).round(2).astype(str) + "%"

rf_display

,Model,Accuracy,Precision,Recall,F1 Score
0,Random Forest - Baseline,93.5%,96.24%,91.27%,93.69%
1,Random Forest - Tuned,93.38%,95.69%,91.61%,93.61%


In [21]:
et = ExtraTreesClassifier(
    random_state=42,
    n_jobs=-1
)

et_param_dist = {
    "n_estimators": [100, 150, 200, 250],
    "max_depth": [None, 10, 15, 20, 25, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

et_search = RandomizedSearchCV(
    estimator=et,
    param_distributions=et_param_dist,
    n_iter=15,
    scoring="f1",
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

et_search.fit(X_train, y_train)

Fitting 3 folds for each of 15 candidates, totalling 45 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",ExtraTreesCla...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'max_depth': [None, 10, ...], 'max_features': ['sqrt', 'log2'], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",15
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichret

In [22]:
print("Best Extra Trees Parameters:")
print(et_search.best_params_)

print("\nBest Cross-Validation F1 Score:")
print(round(et_search.best_score_, 4))

Best Extra Trees Parameters:
{'n_estimators': 250, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None}

Best Cross-Validation F1 Score:
0.9352


In [23]:
best_et = et_search.best_estimator_

y_pred_et_tuned = best_et.predict(X_test)

et_tuned_accuracy = accuracy_score(y_test, y_pred_et_tuned)
et_tuned_precision = precision_score(y_test, y_pred_et_tuned)
et_tuned_recall = recall_score(y_test, y_pred_et_tuned)
et_tuned_f1 = f1_score(y_test, y_pred_et_tuned)

print("Tuned Extra Trees")
print("-----------------")
print("Accuracy :", round(et_tuned_accuracy, 4))
print("Precision:", round(et_tuned_precision, 4))
print("Recall   :", round(et_tuned_recall, 4))
print("F1 Score :", round(et_tuned_f1, 4))

Tuned Extra Trees
-----------------
Accuracy : 0.9318
Precision: 0.951
Recall   : 0.9183
F1 Score : 0.9344


In [24]:
print("\nClassification Report:")
print(classification_report(y_test, y_pred_et_tuned))


Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.95      0.93      2828
           1       0.95      0.92      0.93      3172

    accuracy                           0.93      6000
   macro avg       0.93      0.93      0.93      6000
weighted avg       0.93      0.93      0.93      6000



In [25]:
print("Tuned Extra Trees")
print("-----------------")
print("Accuracy :", round(et_tuned_accuracy, 4))
print("Precision:", round(et_tuned_precision, 4))
print("Recall   :", round(et_tuned_recall, 4))
print("F1 Score :", round(et_tuned_f1, 4))

Tuned Extra Trees
-----------------
Accuracy : 0.9318
Precision: 0.951
Recall   : 0.9183
F1 Score : 0.9344


In [26]:
import joblib
from pathlib import Path

models_dir = Path("../../models")
models_dir.mkdir(parents=True, exist_ok=True)

# Train the selected baseline model one more time
final_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_train, y_train)

# Save model
model_path = models_dir / "random_forest_baseline.pkl"
joblib.dump(final_model, model_path)

print("Final model saved successfully:")
print(model_path)

Final model saved successfully:
..\..\models\random_forest_baseline.pkl


In [27]:
print("Model saved:", model_path.exists())

Model saved: True
